<h3><center> GPT Recreation Using The Multi Head Attention Model </h3>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import matplotlib.pyplot as plt

In [4]:
txt = open('shakespere.txt', 'r', encoding='utf-8')
text = txt.read()
len(text)

1115393

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size, ''.join(chars))

65 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [6]:
stoi = {i:s for s, i in enumerate(chars)}
itos = {s:i for i, s in stoi.items()}

encode = lambda s : [stoi[c] for c in s]
decode = lambda x : ''.join([itos[c] for c in x])

In [7]:
print(encode('hello'))
print(decode(encode('hello')))

[46, 43, 50, 50, 53]
hello


In [8]:
# Embedding character encodes into tensor

data = torch.tensor(encode(text))
data[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [9]:
# Train-test split

k = int(0.8*len(data))
train_dat = data[:k]
test_dat = data[k:]
print(len(train_dat), len(test_dat), len(data))

892314 223079 1115393


In [10]:
# Initialisation

batch_size = 4      # Number of independent squences for batch processing   --> aka batch(B)
block_size = 8      # The context length of the model                       --> aka time(T)
torch.manual_seed(982734)

In [11]:
# Batch dimension

def get_batch(split):
    d = train_dat if split == 'train' else test_dat
    ix = torch.randint(len(d) - block_size, (batch_size, ))
    # print(ix.shape)
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(xb)

tensor([[ 1, 58, 53,  1, 31, 39, 52, 42],
        [ 5, 57,  1, 57, 53, 52,  6,  0],
        [ 1, 58, 53,  1, 58, 46, 63,  1],
        [42, 47, 42,  1, 58, 53,  1, 15]])


In [12]:
# Just to see what the model is reading in a batch

oi = xb.tolist()
xbdecoded = [decode(o) for o in oi]
xbdecoded

[' to Sand', "'s son,\n", ' to thy ', 'did to C']

In [13]:
xb , yb

(tensor([[ 1, 58, 53,  1, 31, 39, 52, 42],
         [ 5, 57,  1, 57, 53, 52,  6,  0],
         [ 1, 58, 53,  1, 58, 46, 63,  1],
         [42, 47, 42,  1, 58, 53,  1, 15]]),
 tensor([[58, 53,  1, 31, 39, 52, 42, 39],
         [57,  1, 57, 53, 52,  6,  0, 13],
         [58, 53,  1, 58, 46, 63,  1, 40],
         [47, 42,  1, 58, 53,  1, 15, 46]]))

In [14]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"{context.tolist()} ------> {target}")

[1] ------> 58
[1, 58] ------> 53
[1, 58, 53] ------> 1
[1, 58, 53, 1] ------> 31
[1, 58, 53, 1, 31] ------> 39
[1, 58, 53, 1, 31, 39] ------> 52
[1, 58, 53, 1, 31, 39, 52] ------> 42
[1, 58, 53, 1, 31, 39, 52, 42] ------> 39
[5] ------> 57
[5, 57] ------> 1
[5, 57, 1] ------> 57
[5, 57, 1, 57] ------> 53
[5, 57, 1, 57, 53] ------> 52
[5, 57, 1, 57, 53, 52] ------> 6
[5, 57, 1, 57, 53, 52, 6] ------> 0
[5, 57, 1, 57, 53, 52, 6, 0] ------> 13
[1] ------> 58
[1, 58] ------> 53
[1, 58, 53] ------> 1
[1, 58, 53, 1] ------> 58
[1, 58, 53, 1, 58] ------> 46
[1, 58, 53, 1, 58, 46] ------> 63
[1, 58, 53, 1, 58, 46, 63] ------> 1
[1, 58, 53, 1, 58, 46, 63, 1] ------> 40
[42] ------> 47
[42, 47] ------> 42
[42, 47, 42] ------> 1
[42, 47, 42, 1] ------> 58
[42, 47, 42, 1, 58] ------> 53
[42, 47, 42, 1, 58, 53] ------> 1
[42, 47, 42, 1, 58, 53, 1] ------> 15
[42, 47, 42, 1, 58, 53, 1, 15] ------> 46


In [15]:
# Feeding the characters in biagram language model that I created in the previous projects

class BiagramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Reading the logits off of the next token from the lookup table and then traning on that model
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx = B
        # targets = T
        # logits = combination of BTC
        logits = self.token_embedding_table(idx)

        # Reshaping logits
        if targets == None:
            loss = None                     # For generating
        else:
            B, T, C = logits.shape          # For training
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is the current context in a batch
        # makes B X T+1, B X T+2, B X T+3 and so on..
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]                           # (B, C)
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)   
            idx = torch.cat((idx, idx_next), dim=1)             # (B, T+1)
        
        return idx

m = BiagramLanguageModel(vocab_size=vocab_size)
logits, loss = m(xb , yb)
print(logits.shape, loss.shape)

torch.Size([32, 65]) torch.Size([])


In [16]:
# Generating from a random model
idx1 = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx1, max_new_tokens=100)[0].tolist()))


Kc;zZTrqj'-CiemzjR:pUr&b'PBhq$!XUgVyXgR.
OSFECiTKVsi$SS
MtJLObcdr GgVQhd3mjv X-FwIzTLuBrkZy,TvscpdUJ


<h3><center> Traning the model</h3>

In [17]:
# Creating an optimiser

optimiser = torch.optim.AdamW(m.parameters(), lr=1e-2)

In [18]:
batch_size = 32
epochs = 10000
for steps in range(epochs):

    # Sample batch of data
    xb , yb = get_batch('train')

    # Evaluate the loss
    logits, loss = m(xb, yb)
    optimiser.zero_grad(set_to_none=True)
    loss.backward()
    optimiser.step()

    print(loss.item())

/home/shaurya/Documents/zth_ak/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


4.615854263305664
4.550692081451416
4.462642669677734
4.4793806076049805
4.586379051208496
4.620168685913086
4.63222599029541
4.515111446380615
4.44936990737915
4.489333629608154
4.4781670570373535
4.542953968048096
4.411465644836426
4.581154823303223
4.338925838470459
4.521949291229248
4.35403299331665
4.396819114685059
4.355238437652588
4.309393405914307
4.385240077972412
4.423508167266846
4.431645393371582
4.371882438659668
4.322421073913574
4.228978157043457
4.338284969329834
4.2148542404174805
4.315412521362305
4.208579063415527
4.312797546386719
4.32181453704834
4.304308891296387
4.256118297576904
4.319732189178467
4.319869041442871
4.101388454437256
4.177739143371582
4.168552398681641
4.1013712882995605
4.239788055419922
4.171548366546631
4.165377140045166
4.095061779022217
4.284010887145996
4.022198677062988
4.178778648376465
4.178906440734863
4.055095195770264
4.142250061035156
4.124187469482422
4.060832500457764
4.056555271148682
3.9990110397338867
4.015390396118164
4.0232181

In [19]:
# Generating from a slightly more trained model
idx1 = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx1, max_new_tokens=500)[0].tolist()))



QULenony t s,

JUELonsthy hamed
LELAD prr yond wim er.
asle
UCEENod
Plimimicckeridoret CA:
KIUTED incand besard ng ege fin mea adiveisinge, thy, Yond maghe isigaknthinstin o ol by teans fay ohtog,
Whakey, yowere;
LAnd, hethavove wins.
feathoth ITounit ondim ap Etimes, mar bewome my hallerofed ayomo ts ayoee m, thr, art to we yo, thomave INE:
NGatixtind and oraindl
Louredwis lisol thind rge w n moveavingr flyowerar omanethasout n cawed! sclleel thate s,
Amourighathoond t, ds e wil chie d thencha


<h2><center>Making self attention Block</h2>

In [22]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)

xbow[:3]

tensor([[[ 5.6640e-01,  1.6632e+00, -2.5062e-01,  1.5695e+00, -1.0643e+00,
          -9.0677e-01, -6.0833e-01,  9.5275e-01, -1.9200e-01,  6.6880e-01,
          -1.1905e+00, -1.1888e+00,  4.6520e-02,  2.5748e-01,  1.5441e+00,
          -3.4084e-01, -2.2192e-01,  7.1438e-01,  9.1835e-01,  1.7265e-02,
           1.5931e+00,  1.7714e+00, -9.5526e-01,  2.2820e-01, -1.1061e+00,
           8.2238e-01, -8.9507e-01,  2.3252e+00,  1.5350e+00, -1.7562e+00,
           2.5055e+00, -5.0894e-01],
         [-4.4149e-01,  8.4779e-01,  2.5591e-01,  8.9377e-01,  4.6202e-02,
           9.6190e-02, -3.8113e-01,  7.6930e-01,  3.3454e-01,  8.5210e-01,
          -1.1368e+00, -3.7164e-01,  1.7979e-01,  5.5410e-01,  1.2238e+00,
          -6.4059e-02, -7.5130e-01,  3.1366e-01,  2.0680e-01, -3.7549e-01,
           3.8034e-01,  1.4191e-01, -4.0803e-01,  6.0982e-01,  3.4156e-01,
           8.4378e-01,  3.3290e-01,  9.6835e-01,  2.3736e-01, -8.9162e-01,
           1.4263e+00, -8.1372e-01],
         [ 2.2635e-01,  1.

In [23]:
# Doing that operation with lower triangular matrix multiplication

wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)

xbow2 = wei @ x
torch.allclose(xbow, xbow2)

False

In [21]:
# What we aim to do is to average the previous few token in a batch (here in the Time T dimension) and then input it into the model

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# Implementing the Key, Query and their interaction using dot product

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)

wei = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))     # Restricts the inter batch communication of the model (encoder block)
wei = F.softmax(wei, dim=-1)

# out = wei @ x
out = wei @ v

out.shape
# What we implemented here is called an decoder block

torch.Size([4, 8, 16])

In [24]:
wei[0]

tensor([1., 0., 0., 0., 0., 0., 0., 0.])

In [25]:
# More accurate recreation of the attention formula as per the paper
# Here we are dividing by the sqrt of the head block size which prevents the extremism of the values in a vector

k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [29]:
B, T, C

(4, 8, 32)

In [39]:
class BatchNorm:

    def __init__(self, dim, eps = 1e-5, momentum = 0.1):

        self.dim = dim
        self.eps = eps
        # self.momentum = momentum
        # self.traning = True

        # Parameters (changes with traning)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # Buffers (changes with momentum)
        # self.running_mean = torch.zeros(dim)
        # self.running_var = torch.ones(dim)

    def __call__(self, x):
        
        # Calculates forward pass
        # if self.traning:
            # if x.ndim == 2:
            #     dim = 0
            # elif x.ndim == 3:
            #     dim = (0,1)
        x_mean = x.mean(1,keepdim=True)
        x_var = x.var(1, keepdim=True)
            # changes for layer norm
        # else:
            # x_mean = self.running_mean
            # x_var = self.running_var

        xhat = (x - x_mean) / torch.sqrt(x_var + self.eps)
        self.out = self.gamma * xhat + self.beta

        # Update the buffers
        # if self.traning:
        #     with torch.no_grad():
        #         self.running_mean = ((1 - self.momentum) * self.running_mean) + (self.momentum * x_mean)
        #         self.running_var = ((1-self.momentum) * self.running_var) + (self.momentum * x_var)
        return self.out
        
    def parameters(self):
        return [self.gamma, self.beta]
    

torch.manual_seed(13372)
module = BatchNorm(100)
x = torch.randn(32,100)
x = module(x)
x.shape

torch.Size([32, 100])

In [40]:
x[:,0].mean(), x[:,0].std()     
# Column norm

(tensor(-7.6976e-05), tensor(0.7465))

In [41]:
x[0, :].mean(), x[0, :].std()
# Row norm

(tensor(7.1526e-09), tensor(1.0000))